# Train Mask2Former **with mask-denoising** on the nanostar data (Kaggle)

Mask-denoising (`src/m2f_denoise.py`, the port of *ref_paper.pdf* — the mask analog
of DN-DETR / MP-Former) improves segmentation of **overlapping** nanostars by giving
every GT instance a dedicated, stably-matched query path during training. It is
**training-only**: the saved checkpoint is a *plain* Mask2Former, so inference via
`src/m2f_pipeline.py` is unchanged.

Unlike the plain `kaggle_train.ipynb`, the denoising decoder **cannot be inlined** —
it reaches into HuggingFace Mask2Former decoder internals (`decoder.mask_predictor`,
`tm.queries_features`, `self.criterion`, ...) that are **grounded against
`transformers` 4.41** (see `docs/mask_denoising.md`). So this notebook:

1. installs a **pinned** `transformers==4.41.2` (NOT Kaggle's preinstalled version), and
2. imports `Mask2FormerDN` + the dataset **from your uploaded `src/` folder**
   instead of inlining them.

It keeps the plain notebook's per-epoch metrics: **val loss** + **COCO mask AP** on full frames.

## 1 · Upload TWO Kaggle Datasets

**(a) The data** — already uploaded as the Kaggle Dataset **`starMeter-data`**
(407 × 4096² JPEGs ≈ 4 GB), with `images/` and `annotations/` inside it:
```
starMeter-data/
  images/        0.jpg, 1.jpg, ...          (the 407 frames)
  annotations/
    train.json   (COCO instances)
    valid.json   (COCO instances)
```

**(b) The code** — upload your repo's **`src/` folder** as a second (tiny) dataset.
The denoising path needs `m2f_denoise.py`, `m2f_dataset.py` and `dataset.py`
(`m2f_dataset` does `from dataset import polygons_to_mask`), so just upload all of `src/`:
```
<code-dataset>/
  src/  m2f_denoise.py  m2f_dataset.py  dataset.py  m2f_pipeline.py  ...
```

Then: **Code → New Notebook → Add Data** (attach BOTH datasets) → **Settings:
Accelerator = GPU (T4/P100), Internet = On** (needed for the `transformers==4.41.2`
install and the `from_pretrained` backbone download) → **File → Upload Notebook**
(this file) → **Run All**.

Keep `SMOKE_TEST = True` for the first run, then set it `False` and Run All again.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.41.2"], check=True)

import transformers
if not transformers.__version__.startswith("4.41"):
    print(f"transformers {transformers.__version__} still loaded in memory; "
          "restarting the kernel so the pinned 4.41.2 takes effect.")
    print(">>> After it restarts, run all cells again. <<<")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    print("transformers", transformers.__version__, "(OK)")

In [ ]:
# --- config -------------------------------------------------------------------
import glob
from pathlib import Path

MODEL             = "facebook/mask2former-swin-tiny-coco-instance"
IMG_SIZE          = 1024     # full crop is downscaled to IMG_SIZE2 (no tiling)
BATCH_SIZE        = 1        # 1024 on swin-tiny fits a 16 GB T4 at batch 1
GRAD_ACCUM        = 2        # effective batch = BATCH_SIZE * GRAD_ACCUM
EPOCHS            = 20
LR                = 5e-5
SAMPLES_PER_EPOCH = 600
NUM_WORKERS       = 2
MIN_PIXELS        = 256      # drop instances smaller than this after cropping
DN_LAMBDA         = 0.2      # fraction of GT mask pixels flipped to build noised dn masks
SCORE_THRESH      = 0.5      # confidence threshold for AP eval
EVAL_EVERY        = 1        # run validation every N epochs
SMOKE_TEST        = True     # first run: 1 epoch, few samples/eval images
OUT_DIR           = Path("/kaggle/working/checkpoints/mask2former-nanostar-dn")

# --- auto-detect the uploaded data + code under /kaggle/input -----------------
def _find(*names):
    for root in ("/kaggle/input", "."):
        for n in names:
            hits = sorted(glob.glob(f"{root}/**/{n}", recursive=True))
            if hits:
                return hits[0]
    return None

TRAIN_JSON = _find("train.json")
VALID_JSON = _find("valid.json", "eval.json")
_sample    = _find("0.jpg", "2.jpg", "1.jpg")
IMAGES_DIR = str(Path(_sample).parent) if _sample else _find("images")
# the folder that contains m2f_denoise.py (your uploaded src/)
_dn        = _find("m2f_denoise.py")
CODE_DIR   = str(Path(_dn).parent) if _dn else None

assert TRAIN_JSON and VALID_JSON and IMAGES_DIR, (
    "Could not locate data. Expected train.json / valid.json and the image folder "
    f"under /kaggle/input. Found: train={TRAIN_JSON} valid={VALID_JSON} images={IMAGES_DIR}")
assert CODE_DIR, (
    "Could not find m2f_denoise.py under /kaggle/input. Upload your repo's src/ "
    "folder as a Kaggle Dataset and attach it (see cell 1).")
print("TRAIN_JSON:", TRAIN_JSON)
print("VALID_JSON:", VALID_JSON)
print("IMAGES_DIR:", IMAGES_DIR)
print("CODE_DIR:  ", CODE_DIR)

In [ ]:
# --- imports (model + dataset come from your uploaded src/) -------------------
import sys
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)   # so `import m2f_dataset` / `import dataset` resolve

import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import Mask2FormerImageProcessor

from m2f_dataset import Mask2FormerDataset, m2f_collate   # repo dataset (supports dn_lambda)
from m2f_denoise import Mask2FormerDN                       # the denoising subclass

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# --- processor, model (Mask2FormerDN), data loaders ---------------------------
processor = Mask2FormerImageProcessor.from_pretrained(MODEL)
processor.size = {"height": IMG_SIZE, "width": IMG_SIZE}  # exact square resize
processor.do_resize = True

# Mask2FormerDN == plain Mask2Former + a training-only denoising group.
model = Mask2FormerDN.from_pretrained(
    MODEL, id2label={0: "nanostar"}, label2id={"nanostar": 0},
    ignore_mismatched_sizes=True).to(device)

if SMOKE_TEST:
    EPOCHS, SAMPLES_PER_EPOCH = 1, 8

# Train with denoising targets (dn_lambda>0 -> dataset emits noised `dn_masks`).
train_ds = Mask2FormerDataset(TRAIN_JSON, IMAGES_DIR, processor, size=IMG_SIZE,
                              samples_per_epoch=SAMPLES_PER_EPOCH, min_pixels=MIN_PIXELS,
                              dn_lambda=DN_LAMBDA)
# Validation never uses denoising (dn is training-only), so dn_lambda=0.
# NB: the repo dataset always applies flip+rot, so val_loss is on augmented crops
# (a rough signal). The deterministic, inference-matching metric is the COCO mask
# AP below, computed on the full 4096^2 frames.
val_ds   = Mask2FormerDataset(VALID_JSON, IMAGES_DIR, processor, size=IMG_SIZE,
                              min_pixels=MIN_PIXELS, dn_lambda=0.0)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, collate_fn=m2f_collate,
                      pin_memory=device == "cuda")
val_dl   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, collate_fn=m2f_collate)
print(f"train crops/epoch: {len(train_ds)}   val crops: {len(val_ds)}   dn_lambda: {DN_LAMBDA}")

In [ ]:
# --- validation: val loss (on crops) + COCO mask AP (full frames) -------------
# Both run model.eval(), so Mask2FormerDN.forward falls through to the plain
# Mask2Former path (no dn) -- exactly what inference does.
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as mask_utils
import numpy as np
from PIL import Image

@torch.no_grad()
def val_loss(model, loader):
    model.eval()
    tot, n = 0.0, 0
    for batch in loader:
        out = model(pixel_values=batch["pixel_values"].to(device),
                    pixel_mask=batch["pixel_mask"].to(device),
                    mask_labels=[m.to(device) for m in batch["mask_labels"]],
                    class_labels=[c.to(device) for c in batch["class_labels"]])
        tot += float(out.loss); n += 1
    return tot / max(n, 1)

@torch.no_grad()
def eval_mask_ap(model, coco_json, images_dir, max_images=None):
    """Run the model on FULL frames (matches inference) and score COCO segm AP."""
    model.eval()
    coco_gt = COCO(coco_json)
    img_ids = coco_gt.getImgIds()
    if max_images:
        img_ids = img_ids[:max_images]
    results = []
    for img_id in tqdm(img_ids, desc="eval", leave=False):
        info = coco_gt.loadImgs(img_id)[0]
        img = Image.open(Path(images_dir) / info["file_name"]).convert("RGB")  # full frame
        enc = processor(images=img, return_tensors="pt").to(device)
        out = model(**enc)
        res = processor.post_process_instance_segmentation(
            out, target_sizes=[(info["height"], info["width"])], threshold=SCORE_THRESH)[0]
        seg = res["segmentation"]
        if seg is None:
            continue
        seg = seg.cpu().numpy()
        for s in res["segments_info"]:
            m = np.asfortranarray((seg == s["id"]).astype(np.uint8))
            rle = mask_utils.encode(m)
            rle["counts"] = rle["counts"].decode()
            results.append({"image_id": img_id, "category_id": 1,
                            "segmentation": rle, "score": float(s["score"])})
    if not results:
        return {"mAP": 0.0, "mAP50": 0.0, "n_pred": 0}
    coco_dt = coco_gt.loadRes(results)
    ev = COCOeval(coco_gt, coco_dt, "segm")
    if max_images:
        ev.params.imgIds = img_ids
    ev.evaluate(); ev.accumulate(); ev.summarize()
    return {"mAP": float(ev.stats[0]), "mAP50": float(ev.stats[1]), "n_pred": len(results)}

In [ ]:
# --- train (with mask-denoising) ----------------------------------------------
opt = torch.optim.AdamW(model.parameters(), lr=LR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
eval_cap = 4 if SMOKE_TEST else None  # smoke: score only a few frames

for epoch in range(EPOCHS):
    model.train()
    running, opt_steps = 0.0, 0
    opt.zero_grad()
    pbar = tqdm(train_dl, desc=f"epoch {epoch + 1}/{EPOCHS}")
    for step, batch in enumerate(pbar):
        out = model(pixel_values=batch["pixel_values"].to(device),
                    pixel_mask=batch["pixel_mask"].to(device),
                    mask_labels=[m.to(device) for m in batch["mask_labels"]],
                    class_labels=[c.to(device) for c in batch["class_labels"]],
                    dn_masks=[d.to(device) for d in batch["dn_masks"]])  # <-- denoising group
        loss = out.loss / GRAD_ACCUM
        loss.backward()
        if (step + 1) % GRAD_ACCUM == 0:
            opt.step(); opt.zero_grad(); opt_steps += 1
        running += float(out.loss)
        pbar.set_postfix(loss=f"{float(out.loss):.4f}")
    train_loss = running / max(len(train_dl), 1)

    # Save a PLAIN Mask2Former every epoch: export_base() strips the dn-only
    # params, so the checkpoint loads unchanged with
    # Mask2FormerForUniversalSegmentation.from_pretrained / src/m2f_pipeline.py.
    model.export_base().save_pretrained(OUT_DIR)
    processor.save_pretrained(OUT_DIR)

    line = f"epoch {epoch + 1}: train_loss {train_loss:.4f}"
    if (epoch + 1) % EVAL_EVERY == 0:
        vl = val_loss(model, val_dl)
        ap = eval_mask_ap(model, VALID_JSON, IMAGES_DIR, max_images=eval_cap)
        line += f" | val_loss {vl:.4f} | mAP {ap['mAP']:.4f} | mAP50 {ap['mAP50']:.4f}"
    print(line, f"  (saved -> {OUT_DIR})")

print("done ->", OUT_DIR)

## Outputs

* `/kaggle/working/checkpoints/mask2former-nanostar-dn/` — a **plain** Mask2Former
  checkpoint (denoising params stripped by `export_base()`), saved **every epoch**.
  Load it with `Mask2FormerForUniversalSegmentation.from_pretrained(...)` or the
  repo's `src/m2f_pipeline.py`. Downloadable from the **Output** tab.

### Tuning
| Variable | Default | Notes |
|---|---|---|
| `DN_LAMBDA` | `0.2` | fraction of GT mask pixels flipped for the noised dn masks (matches `m2f_train.py --lambda-p`) |
| `IMG_SIZE` | `1024` | drop to `768`/`512` if you hit CUDA OOM |
| `BATCH_SIZE` / `GRAD_ACCUM` | `1` / `2` | effective batch = product; raise accum, not batch, on 16 GB |
| `EPOCHS` | `20` | |
| `SAMPLES_PER_EPOCH` | `600` | random crops drawn per epoch |
| `SMOKE_TEST` | `True` | set `False` for the real run |

**Why pinned transformers?** `Mask2FormerDN` reimplements the decoder loop against
`transformers` 4.41 internals; newer releases rename/move those submodules. If the
install in cell 2 is overridden by a preinstalled newer version, restart the kernel
after the pip install. **Internet Off:** add `transformers==4.41.2` as a Kaggle
dataset/wheel and the backbone as a Kaggle model, then point `MODEL` at its local path.